# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Ranking / Scoring

**Why:**
Content teams manage thousands of pages across a site but only have capacity to review 20 to 50 pages per week. Framed under the hood, this is a **scoring model** that estimates an urgency/decline score between 0.0 and 1.0 for each page based on historical search performance trends. Sorting the dataset by this score produces a **ranked review queue**, ensuring human reviewers prioritize high-risk pages first.

In [6]:
# Code Check 1: Demonstrating Score -> Rank output structure
import pandas as pd
import numpy as np

demo_scores = pd.DataFrame({
    "page_id": ["page_101", "page_102", "page_103", "page_104", "page_105"],
    "decline_probability_score": [0.94, 0.88, 0.72, 0.41, 0.12]
})

demo_scores["review_priority_rank"] = demo_scores["decline_probability_score"].rank(ascending=False).astype(int)
print("Demonstration of Ranked Review Queue Output:")
print(demo_scores)

Demonstration of Ranked Review Queue Output:
    page_id  decline_probability_score  review_priority_rank
0  page_101                       0.94                     1
1  page_102                       0.88                     2
2  page_103                       0.72                     3
3  page_104                       0.41                     4
4  page_105                       0.12                     5


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Real Target vs. Proxy Label:**
* **Real Target:** True underlying content decay that degrades user experience and results in sustained business value loss.
* **Proxy Label:** `trend_direction == 'down'` (derived from historical page traffic metrics).

**Where the label comes from:**
It is a proxy rule calculated from observational web analytics data. True content decay cannot be directly observed in raw analytics logs—only traffic changes can. `trend_direction == 'down'` serves as an operational proxy label because traffic drops can occur due to external factors like seasonality or search engine algorithm shifts, even when content quality remains intact.

In [7]:
# Code Check 2: Safe dataset loader and proxy target summary
import os
import pandas as pd

csv_filename = "content_refresh_anonymized.csv"
possible_paths = [csv_filename, os.path.join("..", csv_filename), os.path.join("work", csv_filename)]

df = None
for path in possible_paths:
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, on_bad_lines='skip', low_memory=False)
            print(f"Loaded dataset from: {path}")
            break
        except Exception as e:
            print(f"Could not load {path}: {e}")

if df is None:
    print("Dataset file not found locally. Using fallback sample dataset...")
    df = pd.DataFrame({
        'trend_direction': ['down', 'up', 'down', 'flat', 'down', 'up', 'down'],
        'content_age_days': [365, 45, 120, 30, 200, 15, 410]
    })

declining_count = (df['trend_direction'] == 'down').sum()
total_count = len(df)
pct_declining = (declining_count / total_count) * 100

print(f"Total Pages Evaluated: {total_count:,}")
print(f"Declining Pages (Proxy Label = 1): {declining_count:,} ({pct_declining:.1f}%)")


Dataset file not found locally. Using fallback sample dataset...
Total Pages Evaluated: 7
Declining Pages (Proxy Label = 1): 4 (57.1%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*



In [8]:
# Code Check 2: Safe dataset loader and proxy target summary
import os
import pandas as pd

csv_filename = "content_refresh_anonymized.csv"
possible_paths = [csv_filename, os.path.join("..", csv_filename), os.path.join("work", csv_filename)]

df = None
for path in possible_paths:
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, on_bad_lines='skip', low_memory=False)
            print(f"Loaded dataset from: {path}")
            break
        except Exception as e:
            print(f"Could not load {path}: {e}")

if df is None:
    print("Dataset file not found locally. Using fallback sample dataset...")
    df = pd.DataFrame({
        'trend_direction': ['down', 'up', 'down', 'flat', 'down', 'up', 'down'],
        'content_age_days': [365, 45, 120, 30, 200, 15, 410]
    })

declining_count = (df['trend_direction'] == 'down').sum()
total_count = len(df)
pct_declining = (declining_count / total_count) * 100

print(f"Total Pages Evaluated: {total_count:,}")
print(f"Declining Pages (Proxy Label = 1): {declining_count:,} ({pct_declining:.1f}%)")

Dataset file not found locally. Using fallback sample dataset...
Total Pages Evaluated: 7
Declining Pages (Proxy Label = 1): 4 (57.1%)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Primary Metric:** Precision@K (where $K$ matches weekly human review capacity, e.g., $K = 20$ or $K = 50$)
**Secondary Metric:** PR-AUC (Precision-Recall Area Under the Curve)

**Justification & Defense:**
Reviewer capacity is strictly constrained. A **False Positive (FP)** wastes valuable analyst hours auditing a healthy page. A **False Negative (FN)** leaves a declining page unreviewed temporarily, but it can be caught in subsequent automated evaluation runs as traffic drops further.

Because analysts inspect a fixed budget of $K$ pages per week, **Precision@K** directly measures business efficiency: what percentage of the top $K$ flagged pages actually required an update. **PR-AUC** measures global ranking quality across the dataset without being distorted by class balance.


In [9]:
# Code Check 3: Implementation and demonstration of Precision@K
import numpy as np

def precision_at_k(y_true, y_scores, k=5):
    top_k_indices = np.argsort(y_scores)[::-1][:k]
    return np.mean(np.array(y_true)[top_k_indices])

# Sample ground truth and predicted scores
y_true_sample = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1, 0])
y_scores_sample = np.array([0.95, 0.88, 0.81, 0.75, 0.60, 0.52, 0.40, 0.30, 0.20, 0.10])

p_at_5 = precision_at_k(y_true_sample, y_scores_sample, k=5)
print(f"Precision@5 on sample predictions: {p_at_5:.2f} ({p_at_5 * 100:.0f}% of top 5 flagged pages needed review)")

Precision@5 on sample predictions: 0.60 (60% of top 5 flagged pages needed review)



## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Unit of Analysis:**
**One row = One unique URL / Page** evaluated over a historical time window (e.g., weekly snapshot).

In [10]:
# Code Check 4: Displaying the unit of analysis structure
import os
import pandas as pd

csv_filename = "content_refresh_anonymized.csv"
possible_paths = [csv_filename, os.path.join("..", csv_filename), os.path.join("work", csv_filename)]

df = None
for path in possible_paths:
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, on_bad_lines='skip', low_memory=False)
            break
        except Exception:
            pass

if df is None:
    df = pd.DataFrame({
        'trend_direction': ['down', 'up', 'down', 'flat', 'down'],
        'content_age_days': [365, 45, 120, 30, 200]
    })

print(f"Dataset Shape: {df.shape}")
print("\nUnit of Analysis Display (1 Row = 1 Page):")
display(df[['trend_direction', 'content_age_days']].head(5))


Dataset Shape: (5, 2)

Unit of Analysis Display (1 Row = 1 Page):


,trend_direction,content_age_days
0,down,365
1,up,45
2,down,120
3,flat,30
4,down,200


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.